In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith(('.csv', '.xlsx', '.json')):
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath) / (1024*1024)
            print(f"{size:.1f} MB — {filepath}")
import pandas as pd

files = {
    'TEST': '/content/drive/MyDrive/Cricket Data/test_bbb - 25.xlsx',
    'ODI': '/content/drive/MyDrive/Cricket Data/odi_bbb-25.xlsx',
    'T20I': '/content/drive/MyDrive/Cricket Data/t20_bbb.csv'
}

for format_name, path in files.items():
    print(f"\n{'='*50}")
    print(f"FORMAT: {format_name}")
    print(f"{'='*50}")
    if path.endswith('.csv'):
        df = pd.read_csv(path, nrows=5)
    else:
        df = pd.read_excel(path, nrows=5)

    print(f"Columns: {list(df.columns)}")
    print(f"Sample row:\n{df.iloc[0]}")

218.1 MB — /content/drive/MyDrive/Cricket Data/test_bbb - 25.xlsx
257.6 MB — /content/drive/MyDrive/Cricket Data/odi_bbb-25.xlsx
674.1 MB — /content/drive/MyDrive/Cricket Data/t20_bbb.csv

FORMAT: TEST
Columns: ['Unnamed: 0', 'p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl', 'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal', 'p_out', 'over', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs', 'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs', 'inns_runs', 'inns_wkts', 'inns_balls', 'trail_by', 'lead_by', 'day', 'session', 'target', 'date', 'year', 'ground', 'country', 'winner', 'toss', 'competition', 'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced', 'bowlruns', 'bat_out', 'wagonX', 'wagonY', 'wagonZone', 'line', 'length', 'shot', 'control', 'predscore', 'wprob']
Sample row:
Unnamed: 0                              0
p_match                            226372
inns                                    2
bat                      H

In [8]:
import pandas as pd

# Load ODI data
print("Loading ODI data... (may take a minute)")
odi = pd.read_excel('/content/drive/MyDrive/Cricket Data/odi_bbb-25.xlsx')
print(f"Loaded! Shape: {odi.shape}")

# Basic overview
print(f"\n--- OVERVIEW ---")
print(f"Total balls: {len(odi):,}")
print(f"Unique matches: {odi['p_match'].nunique():,}")
print(f"Unique batters: {odi['bat'].nunique():,}")
print(f"Unique bowlers: {odi['bowl'].nunique():,}")
print(f"Unique teams: {odi['team_bat'].nunique():,}")
print(f"Teams: {sorted(odi['team_bat'].unique())}")
print(f"Year range: {odi['year'].min()} to {odi['year'].max()}")
print(f"Unique grounds: {odi['ground'].nunique():,}")

# Dismissal types
print(f"\n--- DISMISSAL TYPES ---")
print(odi['dismissal'].value_counts())

# Shot types
print(f"\n--- SHOT TYPES ---")
print(odi['shot'].value_counts().head(10))

# Bowling kinds
print(f"\n--- BOWLING KINDS ---")
print(odi['bowl_kind'].value_counts())


Loading ODI data... (may take a minute)


KeyboardInterrupt: 

In [10]:
import pandas as pd
import numpy as np

# --- TOP RUN SCORERS ---
batting = odi.groupby('bat').agg(
    runs=('batruns', 'sum'),
    balls=('ballfaced', 'sum'),
    dismissals=('bat_out', lambda x: (x == True).sum()),
    innings=('p_match', 'nunique')
).reset_index()
batting['sr'] = (batting['runs'] / batting['balls'] * 100).round(2)
batting['avg'] = (batting['runs'] / batting['dismissals'].replace(0, np.nan)).round(2)
batting = batting[batting['balls'] > 500].sort_values('runs', ascending=False)
print("--- TOP 15 RUN SCORERS ---")
print(batting.head(15)[['bat','runs','balls','avg','sr','innings']].to_string())

# --- TOP WICKET TAKERS ---
bowling = odi.groupby('bowl').agg(
    wickets=('out', lambda x: x.sum()),
    runs=('bowlruns', 'sum'),
    balls=('ball', 'count')
).reset_index()
bowling['econ'] = (bowling['runs'] / (bowling['balls']/6)).round(2)
bowling['avg'] = (bowling['runs'] / bowling['wickets'].replace(0, np.nan)).round(2)
bowling = bowling[bowling['balls'] > 300].sort_values('wickets', ascending=False)
print("\n--- TOP 15 WICKET TAKERS ---")
print(bowling.head(15)[['bowl','wickets','runs','econ','avg']].to_string())

# --- TEAM WIN RATES ---
matches = odi.drop_duplicates('p_match')[['p_match','team_bat','team_bowl','winner','year']].copy()
all_teams = pd.concat([
    matches[['p_match','team_bat','winner']].rename(columns={'team_bat':'team'}),
    matches[['p_match','team_bowl','winner']].rename(columns={'team_bowl':'team'})
]).drop_duplicates()
all_teams['won'] = all_teams['team'] == all_teams['winner']
team_stats = all_teams.groupby('team').agg(
    matches=('p_match','nunique'),
    wins=('won','sum')
).reset_index()
team_stats['win_pct'] = (team_stats['wins'] / team_stats['matches'] * 100).round(1)
team_stats = team_stats[team_stats['matches'] >= 20].sort_values('win_pct', ascending=False)
print("\n--- TEAM WIN RATES (min 20 matches) ---")
print(team_stats.to_string())

# --- RUNS PER OVER (scoring trends) ---
over_runs = odi.groupby('over').agg(
    total_runs=('score','sum'),
    balls=('ball','count')
).reset_index()
over_runs['rpo'] = (over_runs['total_runs'] / (over_runs['balls']/6)).round(2)
print("\n--- RUNS PER OVER ---")
print(over_runs[['over','rpo']].to_string())

# --- DISMISSAL TYPES ---
dismissals = odi['dismissal'].value_counts().reset_index()
dismissals.columns = ['type','count']
dismissals = dismissals[dismissals['type'].notna()]
print("\n--- DISMISSALS ---")
print(dismissals.to_string())

--- TOP 15 RUN SCORERS ---
                       bat   runs  balls   avg      sr  innings
1315           Virat Kohli  11783  12632  0.92   93.28      235
674       Kumar Sangakkara  10954  13475  0.79   81.29      265
719               MS Dhoni  10738  12277  0.86   87.46      294
0           AB de Villiers   9580   9476  0.99  101.10      218
1253  Tillakaratne Dilshan   9232  10573  0.85   87.32      258
1029          Rohit Sharma   8934  10055  0.87   88.85      214
1035           Ross Taylor   8321  10077  0.81   82.57      211
721     Mahela Jayawardene   8176   9869  0.81   82.85      252
459            Hashim Amla   8119   9182  0.87   88.42      178
363            Eoin Morgan   7053   7624  0.90   92.51      211
777         Michael Clarke   7039   9052  0.76   77.76      191
1295         Upul Tharanga   6839   9020  0.74   75.82      220
1227           Tamim Iqbal   6834   8783  0.76   77.81      198
754         Martin Guptill   6642   7600  0.85   87.39      175
227          

In [11]:
import pandas as pd
import numpy as np

# Fix batting averages - aggregate at innings level first
innings_agg = odi.groupby(['p_match', 'inns', 'bat']).agg(
    runs=('batruns', 'sum'),
    balls=('ballfaced', 'sum'),
    dismissed=('bat_out', lambda x: 1 if x.any() else 0)  # 1 if got out in this innings
).reset_index()

# Now aggregate across all innings per player
batting = innings_agg.groupby('bat').agg(
    total_runs=('runs', 'sum'),
    total_balls=('balls', 'sum'),
    innings=('runs', 'count'),
    dismissals=('dismissed', 'sum')
).reset_index()

batting['average'] = (batting['total_runs'] / batting['dismissals'].replace(0, np.nan)).round(2)
batting['strike_rate'] = (batting['total_runs'] / batting['total_balls'] * 100).round(2)
batting['not_outs'] = batting['innings'] - batting['dismissals']

# Filter meaningful sample
batting = batting[batting['total_balls'] > 500].sort_values('total_runs', ascending=False)

print("--- TOP 20 BATTING AVERAGES (min 500 balls) ---")
top_avg = batting[batting['innings'] >= 20].sort_values('average', ascending=False).head(20)
print(top_avg[['bat','total_runs','innings','dismissals','not_outs','average','strike_rate']].to_string())

print("\n--- TOP 20 RUN SCORERS WITH CORRECT AVERAGES ---")
print(batting.head(20)[['bat','total_runs','innings','dismissals','not_outs','average','strike_rate']].to_string())

--- TOP 20 BATTING AVERAGES (min 500 balls) ---
                     bat  total_runs  innings  dismissals  not_outs  average  strike_rate
1061  Ryan ten Doeschate        1224       22          22         0    55.64        87.87
1315         Virat Kohli       11783      235         235         0    50.14        93.28
144           Babar Azam        3580       75          75         0    47.73        87.87
494          Imam-ul-Haq        1832       40          40         0    45.80        80.42
459          Hashim Amla        8119      178         178         0    45.61        88.42
1123           Shai Hope        3005       67          67         0    44.85        74.34
1273          Tom Cooper         979       22          22         0    44.50        70.74
762       Matthew Hayden        2488       56          56         0    44.43        82.66
297         David Warner        5142      116         116         0    44.33        95.63
0         AB de Villiers        9580      218       